# 01 · Data audit (shared)
Answers the seven open questions from the proposal with numbers from `src/audit.py`. Figures are in `figures/`.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "results"
load = lambda n: json.loads((R / n).read_text())
pd.set_option("display.precision", 4)

In [2]:
a = load('audit.json')
pd.Series(a['job_status_counts'], name='jobs')

Terminated    732355
Failed        256555
Running        62928
Waiting         3663
Name: jobs, dtype: int64

## Q4: which jobs have a usable runtime label?

In [3]:
pd.DataFrame(a['availability_by_status']).T

,n,job_end_time_present_%,all_task_ends_present_%,exec_time_le0_%,task_start_present_%
Failed,256555.0,9.99,0.38,0.0,100.00
Running,62928.0,1.69,0.37,0.0,100.00
Terminated,732355.0,100.00,100.00,0.0,100.00
Waiting,3663.0,1.23,0.00,0.0,84.33


## Q1/Q2: queue wait and the runtime definition
The gap between job submission and the first task start is 0 for every finished job, so runtime = task span = job duration for Terminated jobs.

In [4]:
pd.Series(a['timing_finished_jobs'])

queue_wait_negative_%                            0.00
queue_wait_median_s                              0.00
queue_wait_p90_s                                 0.00
exec_time_median_s                             660.00
exec_time_p90_s                              12835.00
exec_time_zero_%                                 0.00
job_duration_median_s                          635.00
job_end_minus_task_end_median_s                  0.00
job_end_equals_task_end_%                       76.65
share_of_job_duration_that_is_wait_median        0.00
spearman_wait_vs_exec                             NaN
exec_median_Terminated                         663.00
exec_median_Failed                              19.00
dtype: float64

## Q5: missingness by status (%)

In [5]:
pd.DataFrame(a['task_missing_by_job_status_%'])

,plan_cpu,plan_mem,plan_gpu,gpu_type,inst_num,task_name,start_time,end_time
Failed,0.19,0.19,9.21,9.14,0.00,0.0,0.0,90.45
Running,14.52,14.52,45.56,39.08,0.00,0.0,0.0,98.60
Terminated,0.00,0.00,17.09,17.09,0.00,0.0,0.0,0.00
Waiting,73.19,73.19,97.68,93.17,1.87,0.0,53.5,99.34


## Requested vs assigned GPU type

In [6]:
pd.DataFrame({'gpu_type_spec (requested)': pd.Series(a['gpu_type_spec_values']), 'gpu_type (assigned)': pd.Series(a['gpu_type_values'])})

,gpu_type_spec (requested),gpu_type (assigned)
MISC,19,696281
None,1034399,217738
P100,7653,72732
T4,1099,226923
V100,5750,28797
V100M32,6112,18579


## Q6: drift — failure rate by day

In [7]:
d = pd.DataFrame(a['daily']); d[['day','n','fail_rate']].describe()

,day,n,fail_rate
count,69.0000,69.0000,69.0000
mean,34.0000,15244.0290,0.2624
std,20.0624,8301.4734,0.2054
min,0.0000,9.0000,0.0892
25%,17.0000,13108.0000,0.1507
50%,34.0000,15258.0000,0.1827
75%,51.0000,17889.0000,0.2605
max,68.0000,52467.0000,0.9333


## Leakage table

In [8]:
pd.read_csv(R / 'leakage_table.csv')

,table,column,known_at_submission,used_as,reason
0,job,job_name,yes,key only,Anonymised ID; joins job and task tables. No p...
1,job,inst_id,yes,key only,Job ID; joins the group tag table.
2,job,user,yes,"feature (user_id, top-200 + OTHER) and history...",Known at submission; past-only history per user.
3,job,status,no,label only,Final outcome. Failed vs Terminated is the fai...
4,job,start_time,yes,prediction time; hour and weekday features; sp...,README: submission timestamp. UTC+8 time of da...
5,job,end_time,no,label only (runtime) and history availability ...,Known only after the job ends.
6,task,task_name,yes,"feature (main_role, role_set)","Roles are declared in the submission (ps, work..."
7,task,inst_num,yes,"feature (inst_total, n_tasks)",Requested number of instances.
8,task,status,no,not used,Task outcome; post-submission.
9,task,start_time,no*,runtime label only,Task launch time. Equals job submit time for a...


## The frozen split

In [9]:
s = load('split.json'); {k: s[k] for k in ['frozen_at_utc','cut_day','test_from_day','trace_end_day','counts']}

{'frozen_at_utc': '2026-11-04T13:51:54+00:00',
 'cut_day': 55.15800462962962,
 'test_from_day': 56.15800462962962,
 'trace_end_day': 68.94400462962963,
 'counts': {'dev': {'jobs': 736209,
   'failure_labelled': 685529,
   'runtime_labelled': 545646},
  'late_label': {'jobs': 77, 'failure_labelled': 77, 'runtime_labelled': 58},
  'embargo': {'jobs': 19654,
   'failure_labelled': 17998,
   'runtime_labelled': 14979},
  'test': {'jobs': 295898,
   'failure_labelled': 285306,
   'runtime_labelled': 171672}}}